In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1.shape

(384,)

In [14]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv.shape

(384,)

In [15]:
v1.dot(dv)

np.float32(0.32332402)

In [16]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [17]:
v2.dot(dv)

np.float32(0.019730477)

## 2.3 Embedding Our Dataset

In [27]:
import sys
import os

# Go up one folder to the project root
sys.path.append(os.path.abspath(".."))

In [28]:
from part_1_RAG.ingest import load_faq_data

documents = load_faq_data()

In [29]:
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [34]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vector = model.encode(batch)
    vectors.extend(batch_vector)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1342

In [39]:
import numpy as np

X = np.array(vectors)

In [41]:
X.shape

(1342, 384)

## 2.4 Vector Search

In [50]:
query = "Can I still join the course after the start date?"

v_query = model.encode(query)

In [66]:
scores = X.dot(v_query)

In [67]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294094))

In [68]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [72]:
top5 = np.argsort(scores)[-5:]
top5, scores[top5]

(array([  7, 536, 899, 617,   2]),
 array([0.56009996, 0.6536312 , 0.7192131 , 0.75793695, 0.76294094],
       dtype=float32))

In [73]:
top5 = top5[::-1]
top5, scores[top5]

(array([  2, 617, 899, 536,   7]),
 array([0.76294094, 0.75793695, 0.7192131 , 0.6536312 , 0.56009996],
       dtype=float32))

In [74]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294094
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.75793695
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relat

## 2.5 Vector Search with minsearch

In [75]:
from minsearch import VectorSearch

In [77]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [78]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [84]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [80]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## 2.6 RAG with Vector Search

In [86]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK but ignored by Ollama
)

In [85]:
from part_1_RAG.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from part_1_RAG.rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=ollama_client,
)

In [91]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still sign up.\n\nAccording to the FAQs, you can still join the course. However, please note the following regarding certificates:\n*   **To receive a certificate:** You must submit your project while submissions are still being accepted.\n*   **Registration:** You do not strictly need a confirmation email. You can start learning and submitting homework immediately even without formal registration, as the form is just to gauge interest.'

In [97]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [99]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=ollama_client,
)

In [100]:
query = "I just found out about the program, can I still sign up?"
vector_assistant.rag(query)

'Yes, you can still sign up. According to the provided context:\n\n> "I just discovered the course. Can I still join?\n> A: Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\nAdditionally, note that you can start learning and submitting homework immediately without formally registering, as registration is only used to gauge interest before the start date.'

## 2.7 Vector Search with sqlitesearch

In [114]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [102]:
vs_index.fit(vectors, documents)

In [116]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [117]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [121]:
vs_index.close()